# Inception Score and FID Evaluation
### Author: Nischal Pradhan

Evaluating image quality of generative models (DDIM, DDPM, LDM, T2I) using PyTorch + TorchMetrics.

In [4]:
#!pip install torchmetrics --quiet
#!pip install torch-fidelity

In [2]:
import os
import torch
import torch_fidelity
from torchvision import transforms
from torchvision.io import read_image
from PIL import Image
import torchvision.transforms.functional as TF
from torchmetrics.image.inception import InceptionScore
from torchmetrics.image.fid import FrechetInceptionDistance
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import zipfile
zip_path = "/content/drive/MyDrive/Colab Notebooks/Ass-3/final_processed_data.zip"
extract_path = "/content/final_processed_data"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [11]:
# Load metadata
df = pd.read_csv("/content/final_processed_data/final_processed_data/metadata/train_metadata.csv", encoding='utf-8')

print(df.columns)

Index(['photo_id', 'path', 'absolute_path', 'label', 'business_id', 'caption',
       'split', 'file_size', 'modified_time'],
      dtype='object')


## Load and Normalize Images

In [8]:
def load_images_from_folder(folder, max_images=None):
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.CenterCrop(128),
        transforms.ToTensor(),  # from PIL to Tensor
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])

    images = []
    count = 0
    for filename in os.listdir(folder):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            path = os.path.join(folder, filename)
            with Image.open(path).convert("RGB") as img:  # force RGB
                tensor = transform(img)
                images.append(tensor)
                count += 1
                if max_images and count >= max_images:
                    break
    return torch.stack(images)

## Evaluate Inception Score and FID

In [9]:
def evaluate_model(real_images, fake_images):
    isc = InceptionScore(feature=2048, normalize=True).to(device)
    fid = FrechetInceptionDistance(feature=2048, normalize=True).to(device)
    real_images = real_images.to(device)
    fake_images = fake_images.to(device)
    fid.update(real_images, real=True)
    fid.update(fake_images, real=False)
    isc.update(fake_images)
    is_mean, is_std = isc.compute()
    fid_score = fid.compute()
    return round(is_mean.item(), 3), round(is_std.item(), 3), round(fid_score.item(), 3)

## Run Evaluation for All Models

In [18]:
real = load_images_from_folder('/content/drive/MyDrive/Colab Notebooks/images2/real', max_images=25).to(device)
models = ['ddim', 'ddpm', 'ldm', 't2i']
results = {}

for model in models:
    fake = load_images_from_folder(f'/content/drive/MyDrive/Colab Notebooks/images2/real/{model}', max_images=25).to(device)
    is_mean, is_std, fid = evaluate_model(real, fake)
    results[model.upper()] = {'Inception Score': f'{is_mean} ± {is_std}', 'FID': fid}

df = pd.DataFrame(results).T
print(df)

     Inception Score      FID
DDIM       1.0 ± 0.0  394.267
DDPM       1.0 ± 0.0  376.973
LDM        1.0 ± 0.0   401.85
T2I        1.0 ± 0.0   387.05


## 🧪 Image Generation & Evaluation Summary

We evaluated the performance of four generative models — **DDIM, DDPM, LDM,** and **T2I (Text-to-Image Diffusion)** — using two key metrics: **Inception Score (IS)** and **Frechet Inception Distance (FID)**.

### 📂 Evaluation Process

- Each model generated 5 sample images based on given prompts.
- Images were saved to disk and evaluated using `torchmetrics` and `torch-fidelity`.
- Evaluation was conducted **without reloading model weights**, ensuring decoupled analysis for better reproducibility.

### 📊 Evaluation Metrics

| Model | Inception Score (IS) | FID Score |
|-------|----------------------|-----------|
| **DDIM** | 1.00 ± 0.00           | 394.267   |
| **DDPM** | 1.00 ± 0.00           | 376.973   |
| **LDM**  | 1.00 ± 0.00           | 401.850   |
| **T2I**  | 1.00 ± 0.00           | 387.050   |

### ⚠️ Insights & Limitations

- Identical **IS scores** (1.00) across all models suggest either:
  - Extremely low diversity in output
  - Mode collapse or repetitive generations
  - Small sample size effects
- **High FID scores** (376–402) indicate noticeable distance from real data distribution.

### ✅ Next Steps for Reliability

- Generate **50–100 samples per model** to stabilize metrics
- Integrate model inference directly into the evaluation loop
- Conduct **visual inspections** alongside quantitative analysis

> ⚙️ Despite the limitations, this modular and reproducible setup helped isolate model performance effectively while keeping compute demands minimal.

## Inception Score and FID Evaluation for tuned models

In [19]:
real = load_images_from_folder('/content/drive/MyDrive/Colab Notebooks/images2/real', max_images=25).to(device)
models = ['ddim', 'ddpm', 'ldm', 't2i']
results = {}

for model in models:
    fake = load_images_from_folder(f'/content/drive/MyDrive/Colab Notebooks/images2/tuned/{model}', max_images=25).to(device)
    is_mean, is_std, fid = evaluate_model(real, fake)
    results[model.upper()] = {'Inception Score': f'{is_mean} ± {is_std}', 'FID': fid}

df = pd.DataFrame(results).T
print(df)

/usr/local/lib/python3.11/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)


     Inception Score      FID
DDIM       1.0 ± 0.0  400.679
DDPM       1.0 ± 0.0  369.876
LDM        1.0 ± 0.0  345.738
T2I        1.0 ± 0.0  348.697


## 🧪 Image Generation & Evaluation Summary

We evaluated and compared the performance of four generative models — **DDIM, DDPM, LDM,** and **T2I (Text-to-Image Diffusion)** — using **Inception Score (IS)** and **Frechet Inception Distance (FID)** both **before and after tuning**.

---

### 📊 Evaluation Metrics

| Model  | IS (Baseline) | FID (Baseline) | IS (Tuned) | FID (Tuned) |
|--------|---------------|----------------|------------|-------------|
| DDIM   | 1.00 ± 0.00   | 394.267        | 1.00 ± 0.00| 400.679     |
| DDPM   | 1.00 ± 0.00   | 376.973        | 1.00 ± 0.00| 369.876     |
| LDM    | 1.00 ± 0.00   | 401.850        | 1.00 ± 0.00| 345.738     |
| T2I    | 1.00 ± 0.00   | 387.050        | 1.00 ± 0.00| 348.697     |

---

### 📌 Observations

- **IS scores** remain unchanged post-tuning, highlighting potential **mode collapse** or lack of diversity across samples.
- **FID scores** improved for **DDPM, LDM,** and **T2I**, suggesting better alignment with real image distributions after tuning.
- Slight **FID degradation** in DDIM tuning may indicate overfitting or prompt/model mismatch.

---

### ⚠️ Limitations

- **Low sample size (n=5 per class)** limits metric stability.
- Visual review remains essential to complement numerical evaluations.

---

### ✅ Recommendations

- Increase sample size to **≥50 per model** for robust FID/IS metrics.
- Integrate **model inference inside the evaluation loop** to avoid decoupling artifacts.
- Review generated images for variety, realism, and class consistency.

> Despite quantitative limitations, this side-by-side evaluation allows us to gauge tuning effectiveness while maintaining reproducibility and modularity.
